In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from torchvision.io import read_image
from torchvision.transforms import transforms
from PIL import Image
from tqdm import tqdm

In [2]:
from torchvision.models.vision_transformer import vit_b_16

In [10]:
vgg_model = vit_b_16(weights = True)

In [11]:
vgg_model

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine

In [6]:
# vgg_model.load_state_dict(torch.load('emotion_classifier2.pt'))

In [7]:
vgg_model.heads = nn.Sequential(
    nn.Linear(768, 64),
    nn.Linear(64, 6)
)

In [8]:
vgg_model.load_state_dict(torch.load('emotion_transformer.pt'))

<All keys matched successfully>

In [9]:
import kagglehub

# Download la|test version
path = kagglehub.dataset_download("sujaykapadnis/emotion-recognition-dataset")

print("Path to dataset files:", path)

/home/legion/Documents/mood_analysis/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/legion/.cache/kagglehub/datasets/sujaykapadnis/emotion-recognition-dataset/versions/1


In [10]:
dataset_path = path +'/dataset'
df_path = path +'/data.csv'
df = pd.read_csv(df_path)

In [11]:
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['label'])

In [12]:
normal_transform = transforms.Compose([
     transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [13]:
augmented_transform1 = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(100),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [14]:
augmented_transform2 = transforms.Compose([
    transforms.RandomAutocontrast(),
    transforms.RandomInvert(),
    transforms.RandomGrayscale(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [15]:
class EmotionDataset(Dataset):
    def __init__(self, df, df_path, transform):
        self.df = df 
        self.images  = df['path'].tolist()
        self.labels = df['label'].tolist()
        self.df_path = df_path
        self.transform = transform

    def __len__(self):
        return df.shape[0]
    
    def __getitem__(self, idx):
        
        image = Image.open(self.df_path + '/' + self.images[idx])

        label = self.labels[idx]
        image = self.transform(image)
        label = torch.tensor(label, dtype = torch.long)

        return image, label


In [16]:
emotion_dataset1 = EmotionDataset(df, dataset_path, normal_transform)

In [17]:
dataloader = DataLoader(emotion_dataset1, batch_size = 22, shuffle  = True, num_workers = 8, pin_memory = True)

In [18]:
for p in vgg_model.parameters():
    p.requires_grad = True

In [19]:
# for p in vgg_model.heads.parameters():
    # p.requires_grad = True

In [20]:
lossfn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(vgg_model.parameters(), lr = 1e-5)

In [21]:
epochs = 7

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vgg_model = vgg_model.to(device)
print(f"Using device : {device}")

Using device : cuda


In [23]:
for epoch in range(epochs):
    sum_loss = 0
    for image, label in tqdm(dataloader):
        image, label = image.to(device), label.to(device)
      
        preds = vgg_model(image)
        loss = lossfn(preds, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        sum_loss += loss.item()

    print(f"Total Loss : {sum_loss/(len(dataloader))}")

100%|██████████| 703/703 [07:04<00:00,  1.66it/s]


Total Loss : 0.5664648364800139


100%|██████████| 703/703 [07:02<00:00,  1.66it/s]


Total Loss : 0.3377980820431903


  3%|▎         | 21/703 [00:14<07:34,  1.50it/s]


KeyboardInterrupt: 

In [24]:
torch.save(vgg_model.state_dict(), 'emotion_transformer.pt')

In [ ]:
# df['label'].describe()

count    15453.000000
mean         2.768394
std          1.343195
min          0.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          5.000000
Name: label, dtype: float64